In [ ]:
import base64
import os.path
import re
from email.message import EmailMessage
import mimetypes
import docx
import google.auth
import pandas as pd
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError


In [ ]:
# account = "panirand"
account = "iumrs"

if account == "panirand":
    token_path = "credentials/token_panirand.json"
    creds_path = "credentials/credentials_panirand.json"
elif account == "iumrs":
    token_path = "credentials/token_iumrs.json"
    creds_path = "credentials/credentials_iumrs.json"
else:
    raise ValueError("Invalid account specified. Please choose 'panirand' or 'iumrs'.")

In [ ]:
# If modifying these scopes, delete the file token.json.
# SCOPES = ["https://www.googleapis.com/auth/gmail.readonly"]
# SCOPES = ["https://www.googleapis.com/auth/gmail.compose"]
SCOPES = ["https://www.googleapis.com/auth/gmail.modify"]


def get_auth(token_path=token_path, creds_path=creds_path):
    """Shows basic usage of the Gmail API.
    Lists the user's Gmail labels.
    """
    creds = None
    # The file token.json stores the user's access and refresh tokens, and is
    # created automatically when the authorization flow completes for the first
    # time.
    if os.path.exists(token_path):
        creds = Credentials.from_authorized_user_file(token_path, SCOPES)
    # If there are no (valid) credentials available, let the user log in.
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(creds_path, SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for the next run
        with open(token_path, "w") as token:
            token.write(creds.to_json())

    try:
        # Call the Gmail API
        service = build("gmail", "v1", credentials=creds)
        results = service.users().labels().list(userId="me").execute()
        labels = results.get("labels", [])

        if not labels:
            print("No labels found.")
            return
        print("Labels:")
        for label in labels:
            print(label["name"])

    except HttpError as error:
        # TODO(developer) - Handle errors from gmail API.
        print(f"An error occurred: {error}")


get_auth()

In [ ]:
df = pd.read_excel("output_excel/S01_data.xlsx")
df.head(3)

In [ ]:
from html.parser import HTMLParser
from io import StringIO


class MLStripper(HTMLParser):
    def __init__(self):
        super().__init__()
        self.reset()
        self.strict = False
        self.convert_charrefs = True
        self.text = StringIO()

    def handle_data(self, d):
        self.text.write(d)

    def get_data(self):
        return self.text.getvalue()


def strip_tags_parser(html):
    s = MLStripper()
    s.feed(html)
    return s.get_data()


# html_string = "Some text with <br> a line break and an <img src='pic.jpg'> image."
# cleaned_string = strip_tags_parser(html_string)
# print(cleaned_string)

In [ ]:
# role = "Chair"
role = "Co-chair"

filt_row = df["Role"] == role
df_role = df[filt_row]
df_role["Role"].value_counts()

In [ ]:
if os.path.exists(token_path):
    creds = Credentials.from_authorized_user_file(token_path, SCOPES)
else:
    raise FileNotFoundError(
        f"Token file not found at {token_path}. Please run the authentication flow first."
    )

for data in df_role.to_dict(orient="records")[:1]:
    service = build("gmail", "v1", credentials=creds)
    message = EmailMessage()

    html_filepath = f"{data['html_path']}/{data['html']}"
    if not os.path.exists(html_filepath):
        raise FileNotFoundError(f"HTML file not found at {html_filepath}")
    with open(html_filepath, "r", encoding="utf-8") as f:
        html_body = f.read()
        plain_text_body = strip_tags_parser(html_body)

    # Set plain text version (fallback)
    message.set_content(plain_text_body)

    # Set HTML version
    message.add_alternative(html_body, subtype="html")

    # Attach PDF
    pdf_filepath = f"{data['pdf_path']}/{data['pdf']}"
    if not os.path.exists(pdf_filepath):
        raise FileNotFoundError(f"PDF file not found at {pdf_filepath}")

    type_subtype, _ = mimetypes.guess_type(pdf_filepath)
    if type_subtype:
        maintype, subtype = type_subtype.split("/", 1)
        with open(pdf_filepath, "rb") as fp:
            attachment_data = fp.read()
            # Explicitly set filename so Gmail doesn't show "noname"
            filename = os.path.basename(pdf_filepath)
            message.add_attachment(
                attachment_data,
                maintype=maintype,
                subtype=subtype,
                filename=filename,  # <-- this fixes "noname"
            )

    message["To"] = data["Email"]
    message["From"] = "iumrs.icem2026.thailand@gmail.com"
    message["Subject"] = data["email_subject"]
    message["Cc"] = data["cc_email"]

    # encoded message
    encoded_message = base64.urlsafe_b64encode(message.as_bytes()).decode()

    create_message = {"message": {"raw": encoded_message}}
    # pylint: disable=E1101
    draft = service.users().drafts().create(userId="me", body=create_message).execute()
    
    print(f"Draft created with ID: {draft['id']} for {data['Email']}")
